# EOForestSTAC: TiTiler + Leaflet demo

**Prerequisites**
1. `conda activate eoforeststac-titiler`
2. `python scripts/start_titiler.py` (in a separate terminal)
3. Run this notebook in the same environment

In [1]:
import httpx, json, pystac, ipyleaflet, ipywidgets as widgets
from urllib.parse import urlencode

TITILER = "http://localhost:8000"
CATALOG = "https://s3.gfz-potsdam.de/dog.atlaseo-glm.eo-gridded-data/collections/catalog.json"

## 1. Fetch Zarr URLs from the STAC catalog

In [2]:
catalog = pystac.Catalog.from_file(CATALOG)
dist = catalog.get_child("disturbance-change")

# ── XU_RECOVERY_CURVES (biomass-carbon) ──────────────────────────────────────
xu_item = next(
    catalog.get_child("biomass-carbon")
           .get_child("XU_RECOVERY_CURVES")
           .get_items()
)
xu_zarr = xu_item.assets["zarr"].href
print("XU_RECOVERY_CURVES:", xu_zarr)

# ── HANSEN_GFC (disturbance-change) ──────────────────────────────────────────
hansen_item = next(dist.get_child("HANSEN_GFC").get_items())
hansen_zarr = hansen_item.assets["zarr"].href
print("HANSEN_GFC:", hansen_zarr)

# ── RADD_EUROPE (disturbance-change) ─────────────────────────────────────────
radd_item = next(dist.get_child("RADD_EUROPE").get_items())
radd_zarr = radd_item.assets["zarr"].href
print("RADD_EUROPE:", radd_zarr)

# ── ULS_PRODUCTS (als-lidar) ──────────────────────────────────────────────────
uls_item = next(
    catalog.get_child("als-lidar")
           .get_child("ULS_PRODUCTS")
           .get_items()
)
uls_zarr = {
    "10m":  uls_item.assets["zarr_10m"].href,
    "20m":  uls_item.assets["zarr_20m"].href,
    "100m": uls_item.assets["zarr_100m"].href,
}
print("ULS 10m: ", uls_zarr["10m"])
print("ULS 20m: ", uls_zarr["20m"])
print("ULS 100m:", uls_zarr["100m"])

XU_RECOVERY_CURVES: https://s3.gfz-potsdam.de/dog.atlaseo-glm.eo-gridded-data/collections/XU_RECOVERY_CURVES/XU_RECOVERY_CURVES_v1.0.zarr
HANSEN_GFC: https://s3.gfz-potsdam.de/dog.atlaseo-glm.eo-gridded-data/collections/HANSEN_GFC/HANSEN_GFC_v1.12.zarr
RADD_EUROPE: https://s3.gfz-potsdam.de/dog.atlaseo-glm.eo-gridded-data/collections/RADD_EUROPE/RADD_EUROPE_v1.0.zarr
ULS 10m:  https://s3.gfz-potsdam.de/dog.atlaseo-glm.eo-gridded-data/collections/ULS_PRODUCTS/ULS_HAINICH_10m_v1.0.zarr
ULS 20m:  https://s3.gfz-potsdam.de/dog.atlaseo-glm.eo-gridded-data/collections/ULS_PRODUCTS/ULS_HAINICH_20m_v1.0.zarr
ULS 100m: https://s3.gfz-potsdam.de/dog.atlaseo-glm.eo-gridded-data/collections/ULS_PRODUCTS/ULS_HAINICH_100m_v1.0.zarr


## 2. Inspect XU_RECOVERY_CURVES via TiTiler

In [3]:
# In TiTiler-Xarray v2.x there is no /variables endpoint — use /info
r = httpx.get(f"{TITILER}/info", params={"url": xu_zarr, "variable": "agbmax"})
info = r.json()
print("bounds:", info["bounds"])
print("dimensions:", info.get("dimensions"))
print("band_descriptions:", info.get("band_descriptions"))

bounds: [-180.0, -90.0, 180.0, 90.0]
dimensions: ['disturbance_type', 'y', 'x']
band_descriptions: [['b1', 'dryfire'], ['b2', 'humfire'], ['b3', 'otherdeg'], ['b4', 'regrowth']]


## 3. Map: XU_RECOVERY_CURVES (agbmax by disturbance type)

In TiTiler-Xarray **v2.x** the `sel` parameter uses `key=value` string syntax - not JSON.

In [4]:
DISTURBANCE_TYPES = ["dryfire", "humfire", "otherdeg", "regrowth"]

def xu_tile_url(variable="agbmax", disturbance_type="humfire",
                rescale="0,300", colormap="greens"):
    # v2.x sel syntax: 'key=value'  (NOT json.dumps({...}))
    params = {
        "url":           xu_zarr,
        "variable":      variable,
        "sel":           f"disturbance_type={disturbance_type}",
        "rescale":       rescale,
        "colormap_name": colormap,
        "nodata":        "-9999",
    }
    return f"{TITILER}/tiles/WebMercatorQuad/{{z}}/{{x}}/{{y}}?{urlencode(params)}"


m = ipyleaflet.Map(center=[0, -55], zoom=2, layout=widgets.Layout(height="450px"))
m.add_control(ipyleaflet.FullScreenControl())

tile_layer = ipyleaflet.TileLayer(
    url=xu_tile_url(),
    name="AGB_max",
    opacity=0.85,
    attribution="Xu et al. (2026) doi:10.5281/zenodo.18168285",
)
m.add_layer(tile_layer)

var_picker = widgets.Dropdown(
    options=["agbmax", "b", "c", "d"],
    value="agbmax", description="Variable:", layout=widgets.Layout(width="200px")
)
dt_picker = widgets.Dropdown(
    options=DISTURBANCE_TYPES,
    value="humfire", description="Disturbance:", layout=widgets.Layout(width="200px")
)
cmap_picker = widgets.Dropdown(
    options=["greens", "ylgn", "rdylgn", "viridis", "plasma", "inferno"],
    value="greens", description="Colormap:", layout=widgets.Layout(width="200px")
)
rescale_ranges = {"agbmax": "0,400", "b": "0,0.2", "c": "0.5,4", "d": "0,100"}

def update_layer(_):
    tile_layer.url = xu_tile_url(
        variable=var_picker.value,
        disturbance_type=dt_picker.value,
        rescale=rescale_ranges.get(var_picker.value, "0,300"),
        colormap=cmap_picker.value,
    )

for w in [var_picker, dt_picker, cmap_picker]:
    w.observe(update_layer, "value")

widgets.VBox([widgets.HBox([var_picker, dt_picker, cmap_picker]), m])

## 4. Map — Hansen GFC forest loss year

2D zarr (~30 m global), variable `loss_year` (integer year of first tree-cover loss 2001–2023).

> **Zoom in to level ≥ 7** before tiles appear — at coarser zooms each tile requires > 1 GB from the zarr, which TiTiler rejects. `min_zoom=7` prevents those requests automatically.

In [5]:
def hansen_tile_url(rescale="1,23", colormap="reds"):
    params = {
        "url":           hansen_zarr,
        "variable":      "loss_year",
        "rescale":       rescale,
        "colormap_name": colormap,
        "nodata":        "-9999",
    }
    return f"{TITILER}/tiles/WebMercatorQuad/{{z}}/{{x}}/{{y}}?{urlencode(params)}"


m_hansen = ipyleaflet.Map(center=[-5, -55], zoom=4, layout=widgets.Layout(height="420px"))
m_hansen.add_control(ipyleaflet.FullScreenControl())

hansen_layer = ipyleaflet.TileLayer(
    url=hansen_tile_url(),
    name="Hansen GFC loss year",
    min_zoom=7,          # tiles only requested at zoom ≥ 7
    opacity=0.85,
    attribution="Hansen/UMD/Google/USGS/NASA — doi:10.1126/science.1244693",
)
m_hansen.add_layer(hansen_layer)

cmap_h = widgets.Dropdown(
    options=["reds", "hot", "inferno", "ylorbr", "rdylgn_r"],
    value="reds", description="Colormap:", layout=widgets.Layout(width="180px")
)
note_h = widgets.HTML("<i>Zoom to level 7+ to see tiles</i>")

def update_hansen(_):
    hansen_layer.url = hansen_tile_url(colormap=cmap_h.value)
cmap_h.observe(update_hansen, "value")

widgets.VBox([widgets.HBox([cmap_h, note_h]), m_hansen])

## 5. Map — RADD Europe forest disturbance

Monthly Sentinel-1 based disturbance alerts for Europe. CRS: ETRS89-LAEA (EPSG:3035), 10 m resolution.

Variables:
- `disturbance_occurrence` — binary monthly flag (0/1); select a month with the slider
- `forest_mask` — 0 = non-forest, 1 = forest (static)
- `alert_yydoy` — native alert date code YYddd (static)

> **Zoom to level ≥ 8** — at coarser zoom the 10 m LAEA raster would require too many pixels per tile.

In [6]:
import pandas as pd

RADD_TIMES = pd.date_range("2020-01-01", "2025-12-01", freq="MS")
RADD_LABELS = [t.strftime("%Y-%m") for t in RADD_TIMES]

RADD_VAR_META = {
    "disturbance_occurrence": {"colormap": "reds",   "rescale": "0,1",        "has_time": True},
    "forest_mask":            {"colormap": "greens", "rescale": "0,1",        "has_time": False},
    "alert_yydoy":            {"colormap": "viridis","rescale": "20000,23365","has_time": False},
}

def radd_tile_url(variable="disturbance_occurrence", time_idx=0):
    meta = RADD_VAR_META[variable]
    params = {
        "url":           radd_zarr,
        "variable":      variable,
        "rescale":       meta["rescale"],
        "colormap_name": meta["colormap"],
        "nodata":        "-9999",
    }
    if meta["has_time"]:
        params["isel"] = f"time={time_idx}"
    return f"{TITILER}/tiles/WebMercatorQuad/{{z}}/{{x}}/{{y}}?{urlencode(params)}"


# Centre on central Europe; zoom in to ≥8 to see tiles
m_radd = ipyleaflet.Map(center=[50, 12], zoom=5,
                        layout=widgets.Layout(height="450px"))
m_radd.add_control(ipyleaflet.FullScreenControl())

radd_layer = ipyleaflet.TileLayer(
    url=radd_tile_url(),
    name="RADD disturbance",
    min_zoom=8,
    opacity=0.85,
    attribution="RADD Europe – van der Woude et al. / Wageningen University",
)
m_radd.add_layer(radd_layer)

var_picker_radd = widgets.Dropdown(
    options=list(RADD_VAR_META), value="disturbance_occurrence",
    description="Variable:", layout=widgets.Layout(width="250px"),
)
time_slider = widgets.SelectionSlider(
    options=[(lbl, i) for i, lbl in enumerate(RADD_LABELS)],
    value=0, description="Month:", continuous_update=False,
    layout=widgets.Layout(width="400px"),
)
note_radd = widgets.HTML("<i>Zoom to level 8+ to see tiles</i>")

def update_radd(_):
    v = var_picker_radd.value
    time_slider.disabled = not RADD_VAR_META[v]["has_time"]
    radd_layer.url = radd_tile_url(variable=v, time_idx=time_slider.value)

var_picker_radd.observe(update_radd, "value")
time_slider.observe(update_radd, "value")

widgets.VBox([
    widgets.HBox([var_picker_radd, time_slider]),
    note_radd,
    m_radd,
])

## 6. Map — ULS Products (UAV Laser Scanning)

UAV-derived gridded forest structure for a test region in Hainich National Park, Germany.  
Three resolutions available: **10 m**, **20 m**, **100 m**.  CRS: UTM Zone 32N (EPSG:32632).

Variables include tree height (`h_m_mean`), DBH (`dbh_m_mean`), basal area (`basal_area_m2_ha`), and volume (`volume_m3_ha`).  
> Zoom to level **12+** to see 10 m tiles; 100 m tiles are visible from zoom 9.

In [14]:
ULS_VARIABLES = [
    "h_m_mean", "h_m_median", "h_m_max",
    "dbh_m_mean", "dbh_median", "dbh_m_max",
    "basal_area_m2_ha", "volume_m3_ha",
    "crown_radius_m_mean", "n_segments",
]
ULS_RESCALE = {
    "h_m_mean": "0,40", "h_m_median": "0,40", "h_m_max": "0,50",
    "dbh_m_mean": "0,0.6", "dbh_median": "0,0.6", "dbh_m_max": "0,0.8",
    "basal_area_m2_ha": "0,60", "volume_m3_ha": "0,800",
    "crown_radius_m_mean": "0,4", "n_segments": "0,200",
}

def uls_tile_url(resolution="10m", variable="h_m_mean", colormap="greens"):
    params = {
        "url":           uls_zarr[resolution],
        "variable":      variable,
        "rescale":       ULS_RESCALE.get(variable, "0,100"),
        "colormap_name": colormap,
        "nodata":        "-9999",
    }
    return f"{TITILER}/tiles/WebMercatorQuad/{{z}}/{{x}}/{{y}}?{urlencode(params)}"


# Centre on the Hainich test region (EPSG:4326: ~51.08°N, 10.45°E)
m_uls = ipyleaflet.Map(center=[51.079, 10.453], zoom=13,
                       layout=widgets.Layout(height="450px"))
m_uls.add_control(ipyleaflet.FullScreenControl())

uls_layer = ipyleaflet.TileLayer(
    url=uls_tile_url(),
    name="ULS h_m_mean",
    min_zoom=9,
    opacity=0.85,
    attribution="ULS Products – UAV laser scanning (Hainich, 2022)",
)
m_uls.add_layer(uls_layer)

res_picker = widgets.Dropdown(
    options=["10m", "20m", "100m"], value="10m",
    description="Resolution:", layout=widgets.Layout(width="160px"),
)
var_picker_uls = widgets.Dropdown(
    options=ULS_VARIABLES, value="h_m_mean",
    description="Variable:", layout=widgets.Layout(width="230px"),
)
cmap_picker_uls = widgets.Dropdown(
    options=["greens", "viridis", "plasma", "ylgn", "rdylgn", "blues"],
    value="greens", description="Colormap:", layout=widgets.Layout(width="180px"),
)

def update_uls(_):
    uls_layer.url = uls_tile_url(
        resolution=res_picker.value,
        variable=var_picker_uls.value,
        colormap=cmap_picker_uls.value,
    )

for w in [res_picker, var_picker_uls, cmap_picker_uls]:
    w.observe(update_uls, "value")

widgets.VBox([widgets.HBox([res_picker, var_picker_uls, cmap_picker_uls]), m_uls])